<div style="background-color:#f0f4f8; padding:20px; border-radius:10px;">

# THE DIET PROBLEM

--- 

## Goal of Our Project

- Create an interface where a user can input their age, weight, sex and height as parameters and the program will output an optimised diet at a minimum cost. 
- The program will internally calculate all necessary macronutritional values and output a combination of foods that fulfill all parameters at a minimised cost. 
- The user should then be able to take this list and use it to aid them in their weekly shop.


---

## Rationale 
- We originally encountered a problem similar to this in our Management Science Module, where we were tasked with finding the minimised combination of a given set of 6 foods subject to some dietary constrains by hand using the Simplex Method.
- It was a topic that was of particular interest to use as college students doing our weekly shop on a tight budget (so we could save for more important stuff like our T-Ball ticket 💃🕺)
- However, the method we used was lenghty, complicated and error-prone (if you made a simple mistake, the whole solution came crashing down). It also gave unrealistic outputs that would not realistically sustain a well-balanced diet.
- We decided to try to create an interface that any user, computer wizz or not, can use simply without error or confusing. it would generate a shopping list from the entirity of the available food in a supermarket and be tailored to generate the best combination for any user needs (nutritional or preferential).


<div style="text-align:center; padding:20px;">
    <img src="healthyFood.jpg" width="600" style="border-radius:15px; box-shadow: 0px 4px 15px rgba(0,0,0,0.3);">
    <p style="font-family:Georgia; color:grey; font-size:30px; margin-top:10px;">
        Eat well, spend less 
    </p>
</div>


## Step 1 
### - Importing Essential Libraries


--- 
1. pandas: for reading and manipulating the dataset.
2. numpy: for mathematical operations on arrays etc.
3. re: used to get rid of non numerical data in the data set (like currencies or weights).
4. pulp: found this lib in some of the youtube videos we watched. Its used to solve a LP problem subject to constrains. Perfect for our project goals 🤗.
5. warnings - gets rid of warning messages that may clutter our output.

---

In [2]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "pulp", "pandas", "numpy", "--quiet"])

import pandas as pd
import numpy as np
import pulp
import re
import warnings
warnings.filterwarnings('ignore')


## Step 2:

### -    Loading and Cleaning the Dataset
---

Dataset comes from Tesco UK. Downloaded from Kaggle

In [7]:
data = pd.read_csv("DietProblemData.csv")

---
Imported data was messy - many cells being empty, containing units, using commas instead of decimal points. 
The parse-numeric function below strips anything that isnt a number and returns a clean float value. 

---

In [8]:
def parse_numeric(val): 
    if pd.isna(val): return 0.0          # if the cell is empty (NaN), return 0
    s = str(val).strip()                 
    if s == '' or s == '-': return 0.0
    if s.startswith('<'): return 0.0     # nutriens in only trace amounts are returned as 0 too
    s = re.sub(r'\s*\(.*?\)', '', s)     
    s = re.sub(r'[gG€%]', '', s)         
    s = s.replace(',', '.').strip()      
    try: return float(s)                 
    except: return 0.0                   # if it cant be converted to a clean float, return 0

---

Building the clean data frame.
This creates a clean table with defined rows (food product) and columns (nutritional values & price)

---

In [9]:
df = pd.DataFrame() #cleaned data stored in df

df['name']           = data['Product Name'].str.strip() #change name and get rid of space
df['price_eur']      = data['Price (€)'].apply(parse_numeric) #runs through parse to get a clean number (.apply calls the function row by row)
df['pack_g']         = data['Pack Size (g)'].apply(parse_numeric)
df['servings']       = data['Serving Size (servings)'].apply(parse_numeric)
df['energy_kcal']    = data['Energy (kcal)'].apply(parse_numeric)
df['fat_g']          = data['Fat (g)'].apply(parse_numeric)
df['saturates_g']    = data['Saturates (g)'].apply(parse_numeric)
df['carbs_g']        = data['Carbohydrates (g)'].apply(parse_numeric)
df['sugars_g']       = data['Sugars (g)'].apply(parse_numeric)
df['fibre_g']        = data['Fibre (g)'].apply(parse_numeric)
df['protein_g']      = data['Protein (g)'].apply(parse_numeric)
df['salt_g']         = data['Salt (g)'].apply(parse_numeric)



df = df[(df['price_eur'] > 0) & (df['pack_g'] > 0) & (df['servings'] > 0)].copy() #gets rid of rows where size is 0
df = df.reset_index(drop=True) #resets row number
df['cost_per_serving'] = df['price_eur'] / df['servings']
df['serving_g']        = df['pack_g'] / df['servings'] #weight of 1 serving in grams

print(f"{len(df)} usable food products")
display(df[['name','price_eur','servings','cost_per_serving','energy_kcal',
            'fat_g','carbs_g','protein_g','fibre_g','salt_g']].head())

162 usable food products


,name,price_eur,servings,cost_per_serving,energy_kcal,fat_g,carbs_g,protein_g,fibre_g,salt_g
0,Crispy Pancakes Beef & Onion,2.57,4.0,0.642500,274.0,15.0,26.0,8.2,1.1,0.72
1,Fish Seasoning,0.88,4.0,0.220000,253.0,3.0,46.5,6.0,2.5,22.01
2,Hearty Food Co Mac 'N' Cheese,0.76,1.0,0.760000,121.0,2.7,19.3,4.4,1.1,0.62
3,Dijon Mustard,2.34,37.0,0.063243,128.0,9.2,3.7,6.2,2.7,6.20
4,Limited Edition Milk Shake Mix Lime Flavoured,2.34,10.0,0.234000,6.0,0.5,0.4,0.0,0.0,0.10


In [1]:
print(" 👇 Enter your details below 👇 \n")

AGE       = int(input("Enter your age (years): "))
SEX       = input("Enter your sex (male/female): ").strip().lower()

# the user can choose if they want to put their measurements in metric or impreial
height_unit = input("Would you like to enter your height in cm or feet? (cm/feet): ").strip().lower()
if height_unit == 'feet':
    feet   = float(input("Enter feet: "))
    inches = float(input("Enter inches (enter 0 if none): "))
    HEIGHT_CM = round((feet * 30.48) + (inches * 2.54), 1)
    print(f"  Converting to cm: {HEIGHT_CM}cm")
else:
    HEIGHT_CM = float(input("Enter your height (cm): "))

    
weight_unit = input("Would you like to enter your weight in kg or lbs? (kg/lbs): ").strip().lower()
if weight_unit == 'lbs':
    weight_lbs = float(input("Enter your weight (lbs): "))
    WEIGHT_KG  = round(weight_lbs * 0.453592, 1)
    print(f"  Converting to kg: {WEIGHT_KG}kg")
else:
    WEIGHT_KG = float(input("Enter your weight (kg): "))

# we put activity levels as pick an option instead of type to avoid typos which would lead to errors
print("\nActivity level options:")
print("  A: Inactive       (desk job, little exercise)")
print("  B: Lightly Active  (light exercise 1-3 days/week)")
print("  C: Moderately Active (moderate exercise 3-5 days/week)")
print("  D: Very Active     (hard exercise 6-7 days/week)")
print("  E: Extra Active    (very hard exercise / physical job)")

activity_choice = input("Enter your activity level (A/B/C/D/E): ").strip().upper()

activity_map = {
    'A': 'inactive',
    'B': 'lightly_active',
    'C': 'moderately_active',
    'D': 'very_active',
    'E': 'extra_active'
}
ACTIVITY = activity_map.get(activity_choice, 'moderately_active')
print(f"  Selected: {ACTIVITY.replace('_',' ').title()}")

print("-" * 28)
print(f"""
  ⚡️ Here are the details you entered ⚡️
       
  Age      : {AGE} years old 
  Sex      : {SEX.capitalize()}
  Height   : {HEIGHT_CM}cm
      
  Weight   : {WEIGHT_KG}kg
  Activity : {ACTIVITY.replace('_',' ').title()}
""")
print("-" * 28)

 👇 Enter your details below 👇 



Enter your age (years):  20
Enter your sex (male/female):  male
Would you like to enter your height in cm or feet? (cm/feet):  feet
Enter feet:  6
Enter inches (enter 0 if none):  0


  Converting to cm: 182.9cm


Would you like to enter your weight in kg or lbs? (kg/lbs):  kg
Enter your weight (kg):  70



Activity level options:
  A: Inactive       (desk job, little exercise)
  B: Lightly Active  (light exercise 1-3 days/week)
  C: Moderately Active (moderate exercise 3-5 days/week)
  D: Very Active     (hard exercise 6-7 days/week)
  E: Extra Active    (very hard exercise / physical job)


Enter your activity level (A/B/C/D/E):  D


  Selected: Very Active
----------------------------

  ⚡️ Here are the details you entered ⚡️
       
  Age      : 20 years old 
  Sex      : Male
  Height   : 182.9cm
      
  Weight   : 70.0kg
  Activity : Very Active

----------------------------


In [2]:
if SEX.lower() == 'male':
    bmr = 10 * WEIGHT_KG + 6.25 * HEIGHT_CM - 5 * AGE + 5
else:
    bmr = 10 * WEIGHT_KG + 6.25 * HEIGHT_CM - 5 * AGE - 161

activity_factors = {
    'inactive': 1.2, 'lightly_active': 1.375,
    'moderately_active': 1.55, 'very_active': 1.725, 'extra_active': 1.9
}
tdee = bmr * activity_factors.get(ACTIVITY, 1.55)

protein_factor = 1.2 if ACTIVITY in ('moderately_active','very_active','extra_active') else 0.8
protein_min = round(protein_factor * WEIGHT_KG, 1)
protein_max = round(2.0 * WEIGHT_KG, 1)
fat_min     = round(0.20 * tdee / 9, 1)
fat_max     = round(0.35 * tdee / 9, 1)
sat_max     = round(0.10 * tdee / 9, 1)
carb_min    = round(0.45 * tdee / 4, 1)
carb_max    = round(0.65 * tdee / 4, 1)
sugar_max   = round(0.10 * tdee / 4, 1)
fibre_min   = 30.0 if AGE >= 18 else 25.0
salt_max    = 6.0

print(f"""
  Based on your details, your estimated calorie needs are:

  At complete rest: {bmr:.0f} kcal/day 
  Accounting for activity level : {tdee:.0f} kcal/day 

  Overall daily nutrient targets are:
  
  Energy      : {tdee:.0f} kcal
  Protein     : between {protein_min}g and {protein_max}g
  Fat         : between {fat_min}g and {fat_max}g
  Saturates   : no more than {sat_max}g
  Carbs       : between {carb_min}g and {carb_max}g
  Sugars      : no more than {sugar_max}g
  Fibre       : at least {fibre_min}g
  Salt        : no more than {salt_max}g
""")


  Based on your details, your estimated calorie needs are:

  At complete rest: 1748 kcal/day 
  Accounting for activity level : 3016 kcal/day 

  Overall daily nutrient targets are:
  
  Energy      : 3016 kcal
  Protein     : between 84.0g and 140.0g
  Fat         : between 67.0g and 117.3g
  Saturates   : no more than 33.5g
  Carbs       : between 339.2g and 490.0g
  Sugars      : no more than 75.4g
  Fibre       : at least 30.0g
  Salt        : no more than 6.0g

